# Mortgage Mathematics & Amortization

From-scratch implementations of mortgage payment calculations, amortization schedules, prepayment analysis, adjustable-rate mechanics, refinancing decisions, and weighted average life — all built with NumPy.

---

### What You Will Learn

This notebook takes you from the fundamental annuity formula through the full landscape of mortgage mathematics. By the end, you will understand:

- How a single formula determines your monthly payment for 30 years
- Why your early payments barely reduce your loan balance
- The precise mathematics behind prepayment savings
- How adjustable-rate mortgages work under the hood, including caps and resets
- A rigorous NPV framework for refinancing decisions
- How Wall Street models prepayment risk using the PSA model and weighted average life

### Why This Matters

A mortgage is, for most people, the single largest financial transaction of their lives. The difference between a well-understood mortgage decision and a naive one can easily be $100,000 or more over the life of the loan. Beyond personal finance, mortgage mathematics underpins the **$12 trillion U.S. mortgage-backed securities (MBS) market** — making these concepts essential for anyone working in fixed income, structured products, or risk management.

### Prerequisites

- Basic algebra and geometric series
- Familiarity with present value and discount factors
- Python/NumPy (for following the code)

### Notation

| Symbol | Meaning |
|--------|---------|
| $PV$ | Present value / loan principal |
| $PMT$ | Fixed monthly payment |
| $r$ | Monthly interest rate ($= r_{\text{annual}} / 12$) |
| $n$ | Total number of monthly payments ($= \text{years} \times 12$) |
| $B_k$ | Outstanding balance after payment $k$ |
| $I_k$ | Interest portion of payment $k$ |
| $P_k$ | Principal portion of payment $k$ |

### How to Use This Notebook

This notebook is structured as a progressive journey through mortgage mathematics. Each section builds on the previous one:

- **Sections 3-5** cover the core mechanics that every borrower should understand: how payments are calculated, how amortization works, and what the numbers actually look like over 30 years.
- **Section 6** addresses the most actionable personal finance question: when and how much to prepay.
- **Sections 7-8** cover the two major mortgage decisions beyond the initial purchase: choosing between fixed and adjustable rates, and deciding whether to refinance.
- **Section 9** bridges personal finance and institutional finance, showing how the same mortgage math applies to the multi-trillion-dollar MBS market.

Each section follows the same pattern:
1. **Theory and intuition** — the "why" before the "how"
2. **Mathematical derivation** — rigorous but accessible
3. **Worked examples** — concrete numbers you can verify by hand
4. **Code implementation** — from-scratch NumPy, no black boxes
5. **Visualization and interpretation** — what the numbers mean in practice

> **Key Concept:** Throughout this notebook, we use a running example of a **$400,000 mortgage at 6.5% for 30 years**. This represents a typical U.S. home purchase. All worked examples, tables, and visualizations reference this baseline so you can build cumulative intuition.

### The Mathematics You Need

The entire notebook rests on one central formula and its consequences:

$$PMT = PV \cdot \frac{r(1+r)^n}{(1+r)^n - 1}$$

Everything else — amortization schedules, prepayment analysis, ARM mechanics, refinancing decisions, and WAL — is a variation on this theme. If you understand where this formula comes from and what each variable means, the rest follows naturally.

### A Note on Conventions

- All interest rates are quoted as annual rates (APR) unless otherwise stated. We always convert to monthly rates for calculations: $r_{\text{monthly}} = r_{\text{annual}} / 12$.
- Dollar amounts are rounded in the text for readability, but the code computes with full precision.
- "Extra payment" always means an additional amount applied directly to principal reduction, above the required monthly payment.
- We ignore taxes, insurance, PMI, and other real-world costs to focus on the core mathematics. These factors matter in practice but do not change the fundamental formulas.

### What Makes This Notebook Different

Many online mortgage calculators give you a monthly payment number but hide the mathematics. This notebook shows you *exactly* how every number is computed, from the geometric series derivation of the annuity formula through the PSA prepayment model. When you finish, you will not just know the answers — you will understand *why* the answers are what they are, and you will be able to adapt the analysis to any mortgage scenario you encounter.

## 1. Motivation

### The Biggest Purchase of Your Life

A mortgage is the **largest financial obligation** most people will ever undertake. A $400,000 mortgage at 6.5% for 30 years will cost you over $510,000 in interest alone — more than the house itself. Understanding the mathematics behind mortgage payments is not just an academic exercise; it is essential for making some of the most important financial decisions of your life:

- Should you choose a 15-year or 30-year term?
- How much do extra monthly payments really save?
- When does refinancing make financial sense?
- Should you take a lower ARM rate or a higher fixed rate?

The core mathematics rests on the **annuity formula** — an elegant result from the geometric series that determines how a fixed payment simultaneously covers interest and repays principal.

### A Quick Reality Check

Suppose you buy a home for $400,000 with a 30-year mortgage at 6.5%. Here is what happens over the life of the loan:

| Item | Amount |
|------|--------|
| Purchase price | $400,000 |
| Total interest paid | ~$510,000 |
| Total amount paid | ~$910,000 |
| Interest-to-principal ratio | 1.28x |

You read that correctly: **you pay more in interest than the house cost**. At 8%, the ratio climbs to 1.64x — nearly three times the purchase price leaves your bank account. This is not a defect of the financial system; it is a mathematical consequence of compound interest over long time horizons. Understanding *why* this happens, and what you can do about it, is the purpose of this notebook.

### Who Else Cares About Mortgage Math?

Beyond personal finance, mortgage mathematics is critical for the **$12 trillion US mortgage-backed securities (MBS) market**. Banks, hedge funds, and insurance companies that buy MBS need to model prepayment speeds, weighted average life, and interest rate risk — all topics we cover here.

In Section 9, we will see how the PSA prepayment model and weighted average life (WAL) connect individual mortgage decisions to institutional portfolio risk management.

> **Key Concept:** A mortgage is fundamentally an **annuity** — a series of equal payments that simultaneously pay interest on the outstanding balance and reduce the principal. The magic of the annuity formula is that a single fixed monthly payment accomplishes both goals, with the split between interest and principal shifting over time.

### Roadmap

| Section | Topic | Key Question |
|---------|-------|-------------|
| 3 | Fixed-rate mortgage | What is my monthly payment and total cost? |
| 4-5 | Amortization | Why are early payments mostly interest? |
| 6 | Prepayment | How much do extra payments really save? |
| 7 | ARMs | When is a variable rate worth the risk? |
| 8 | Refinancing | When should I replace my mortgage? |
| 9 | WAL & PSA | How do investors model mortgage pools? |

### The Historical Context

Mortgage mathematics as we know it is relatively modern. The fully amortizing fixed-rate mortgage was introduced in the 1930s as part of the New Deal response to the Great Depression. Before that, most mortgages were "balloon" loans: you paid interest only for 5-10 years, then owed the entire principal as a lump sum. If you could not pay or refinance, you lost the house.

The innovation of amortization — spreading principal repayment across hundreds of equal monthly payments — made homeownership accessible to the middle class. Combined with government guarantees (FHA, VA, Fannie Mae, Freddie Mac), it transformed American society and created the suburban landscape we know today.

The secondary market for mortgages (MBS) was created in 1970 when Ginnie Mae issued the first mortgage pass-through security. This innovation allowed banks to sell their mortgage portfolios to investors, freeing up capital for new lending. Today, the MBS market is the second-largest fixed-income market in the world, behind only U.S. Treasuries.

### The Mathematical Toolkit

The mathematics in this notebook draws from several areas:

| Mathematical Tool | Where It Appears |
|-------------------|-----------------|
| Geometric series | Annuity formula derivation (Section 3) |
| Recursive equations | Amortization schedule (Section 4) |
| Optimization (NPV) | Prepayment and refinancing decisions (Sections 6, 8) |
| Probability (CPR/SMM) | PSA prepayment model (Section 9) |
| Weighted averages | Weighted average life (Section 9) |

None of these require advanced mathematics — everything is accessible with algebra and basic calculus intuition.

### The Scale of the Mortgage Market

To appreciate why mortgage mathematics matters beyond your own front door, consider these figures:

| Metric | Value |
|--------|-------|
| Total U.S. mortgage debt outstanding | ~$12.5 trillion (2024) |
| Total U.S. MBS market | ~$12 trillion |
| Annual mortgage originations | ~$1.5-2.5 trillion |
| Number of U.S. mortgages | ~50 million |
| Average mortgage balance | ~$250,000 |
| MBS as % of U.S. bond market | ~22% |

The MBS market is larger than the corporate bond market and second only to U.S. Treasuries. Every basis point of interest rate change moves billions of dollars in value across this market. The mathematics we develop here — particularly amortization, prepayment modeling, and WAL — is the foundation on which this market operates.

## 2. Setup

We use only NumPy, SciPy, and Matplotlib — no specialized finance libraries. Every calculation is built from first principles so you can see exactly what is happening under the hood.

We keep the setup minimal and standard. A fixed random seed ensures reproducibility, and we define a consistent color palette for all visualizations throughout the notebook.

The tolerance constants (`ATOL`, `RTOL`) are available for numerical verification but are not used in the main calculations since our formulas are analytical (closed-form) rather than iterative.

The color scheme is chosen for clarity: **steelblue** for principal-related quantities, **coral** for interest-related quantities, **seagreen** for savings and positive outcomes, and **gold** for accent/highlight. This consistent mapping helps you read the charts intuitively throughout the notebook.

In [ ]:
%matplotlib inline
import numpy as np
from scipy import linalg, optimize
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
rng = np.random.default_rng(SEED)

ATOL = 1e-10
RTOL = 1e-6

PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

## 3. Fixed-Rate Mortgage

### The Core Question

You borrow $400,000 to buy a house at 6.5% interest for 30 years. What is your monthly payment?

This is the fundamental mortgage calculation, and it comes directly from the **present value of an annuity** formula. Before we jump to the formula, let us build intuition for *why* it works.

### Building Intuition: What Does a Lender Require?

Think about it from the bank's perspective. The bank gives you $400,000 today. In return, they want a stream of equal monthly payments that, when discounted back to today at the agreed interest rate, are worth exactly $400,000. This is the **present value equivalence** principle:

$$\text{Value of lump sum today} = \text{Present value of all future payments}$$

The bank does not care whether you pay mostly interest at first or mostly principal — the annuity formula automatically handles this. All the bank cares about is that the present value of your payment stream equals the loan amount.

### Derivation from First Principles

**Step 1: Present value of a single future payment.** A payment of $PMT$ received $k$ months from now, discounted at monthly rate $r$, is worth:

$$PV_k = \frac{PMT}{(1+r)^k}$$

**Step 2: Sum all payments.** The total present value of $n$ equal payments is:

$$PV = \sum_{k=1}^{n} \frac{PMT}{(1+r)^k} = PMT \sum_{k=1}^{n} \frac{1}{(1+r)^k}$$

**Step 3: Evaluate the geometric series.** The sum $\sum_{k=1}^{n} x^k$ with $x = \frac{1}{1+r}$ is a standard geometric series:

$$\sum_{k=1}^{n} x^k = x \cdot \frac{1 - x^n}{1 - x} = \frac{1}{1+r} \cdot \frac{1 - (1+r)^{-n}}{1 - \frac{1}{1+r}} = \frac{1 - (1+r)^{-n}}{r}$$

**Step 4: The annuity present value formula.** Substituting back:

$$PV = PMT \cdot \frac{1 - (1+r)^{-n}}{r}$$

**Step 5: Solve for PMT.** The monthly payment is:

$$\boxed{PMT = PV \cdot \frac{r}{1 - (1+r)^{-n}} = PV \cdot \frac{r(1+r)^n}{(1+r)^n - 1}}$$

Both forms are equivalent. The first is slightly more intuitive; the second avoids computing a negative exponent.

> **Key Concept:** The annuity formula is a direct consequence of the geometric series. It answers: "What fixed payment, repeated $n$ times and discounted at rate $r$, has a present value equal to $PV$?"

### Worked Example: $400,000 at 6.5% for 30 Years

Let us walk through this step by step with actual numbers.

**Given:** $PV = \$400{,}000$, annual rate = 6.5%, term = 30 years.

**Step 1 — Monthly rate:**
$$r = \frac{0.065}{12} = 0.0054167$$

**Step 2 — Number of months:**
$$n = 30 \times 12 = 360$$

**Step 3 — Compute the growth factor:**
$$(1+r)^n = (1.0054167)^{360} = 6.9913$$

This number tells us that $1 invested at the monthly rate for 360 months grows to $6.99. It captures the full compounding effect over 30 years.

**Step 4 — Apply the formula:**
$$PMT = 400{,}000 \times \frac{0.0054167 \times 6.9913}{6.9913 - 1} = 400{,}000 \times \frac{0.03788}{5.9913} = \$2{,}528.27$$

**Step 5 — Total cost:**
$$\text{Total paid} = 2{,}528.27 \times 360 = \$910{,}177$$
$$\text{Total interest} = 910{,}177 - 400{,}000 = \$510{,}177$$

> **Common Mistake:** Using the annual rate directly instead of dividing by 12. If you plug 0.065 into the formula instead of 0.065/12, you get a wildly wrong answer. Always convert to the rate per compounding period.

### The Shocking Truth About 30-Year Mortgages

At 6.5% for 30 years, you pay **1.28x** the original loan amount in interest alone. At 8%, the ratio climbs to **1.64x** — you pay nearly three times the house price. This is why understanding mortgage math can save you hundreds of thousands of dollars.

Why is the interest so high? Because the balance remains large for most of the loan's life. In the early years, you are paying interest on nearly the full $400,000. The principal barely decreases. We will see exactly why in the amortization section.

### 15-Year vs 30-Year: The Great Debate

This is one of the most common personal finance decisions. Let us compare the numbers directly.

| Feature | 15-Year | 30-Year |
|---------|---------|---------|
| Monthly payment | Higher (~$3,484 at 6.5%) | Lower (~$2,528 at 6.5%) |
| Total interest | Much less (~$227K) | Much more (~$510K) |
| Interest savings | ~$283K | — |
| Flexibility | Less (locked into higher payment) | More (lower required payment) |
| Rate | Usually 0.5-0.75% lower | Usually higher |

The 15-year mortgage saves enormous interest, but requires a 38% higher monthly payment. The financially optimal choice depends on your cash flow needs and what you would do with the payment difference.

**The nuanced view:** If you take the 30-year mortgage but invest the $956/month payment difference in an index fund earning 8% on average, you might come out ahead after 30 years — but you are taking on market risk. The 15-year mortgage gives you a guaranteed "return" equal to the mortgage rate. There is no single right answer; it depends on your risk tolerance, tax situation, and financial goals.

> **CFA Exam Tip:** The annuity present value formula appears throughout the CFA curriculum — in bond pricing (coupon streams), pension valuation, and lease analysis. Mastering it here pays dividends across all fixed-income topics.

> **Key Concept:** The interest you pay on a mortgage is *not* simply principal times rate times years. Because you are paying down the principal over time, the actual interest cost depends heavily on the amortization schedule. Early payments are mostly interest; late payments are mostly principal.

> **Common Mistake:** Comparing mortgages only by monthly payment. A lower payment (longer term) usually means dramatically more total interest. Always consider the total cost of borrowing.

### Implementation

The code below computes monthly payments and total interest across various rates and terms. We implement the annuity formula directly, with a special case for zero interest rates (where the formula reduces to simple division).

Notice how the comparison table reveals the *multiplicative* interaction between rate and term — doubling either one roughly doubles the total interest.

**What to look for in the output:**
- The monthly payment column shows the cash flow impact of each rate/term combination
- The total interest column shows the lifetime cost — this is the number that most people dramatically underestimate
- The interest/principal ratio tells you how many dollars of interest you pay per dollar borrowed

For our baseline case ($400,000 at 6.5% for 30 years), the monthly payment of ~$2,528 looks manageable. But the total interest of ~$510,000 means you are paying $1.28 for every dollar you borrowed. At 8% for 30 years, it rises to $1.64 per dollar borrowed.

In [ ]:
def monthly_payment(principal, annual_rate, years):
    """Compute fixed monthly mortgage payment.
    
    Parameters
    ----------
    principal : float — loan amount
    annual_rate : float — annual interest rate (decimal)
    years : int — loan term in years
    
    Returns
    -------
    float — monthly payment
    """
    r = annual_rate / 12.0  # monthly rate
    n = years * 12           # total months
    
    if abs(r) < 1e-15:
        return principal / n
    
    return principal * r * (1 + r) ** n / ((1 + r) ** n - 1)


def total_interest(principal, annual_rate, years):
    """Total interest paid over the life of the mortgage."""
    pmt = monthly_payment(principal, annual_rate, years)
    return pmt * years * 12 - principal


# --- Example ---
principal = 400_000
rate = 0.065
term = 30

pmt = monthly_payment(principal, rate, term)
total_paid = pmt * term * 12
total_int = total_interest(principal, rate, term)

print(f"Mortgage: ${principal:,.0f} at {rate:.2%} for {term} years")
print(f"Monthly Payment:   ${pmt:>10,.2f}")
print(f"Total Paid:        ${total_paid:>10,.0f}")
print(f"Total Interest:    ${total_int:>10,.0f}")
print(f"Interest / Principal ratio: {total_int / principal:.2f}x")

# Compare terms
print("\n\nComparison Across Terms and Rates")
print("-" * 70)
print(f"{'Rate':>6s} {'Term':>6s} {'Monthly PMT':>14s} {'Total Interest':>16s} {'Interest/Prin':>14s}")
print("-" * 70)
for r in [0.04, 0.05, 0.06, 0.065, 0.07, 0.08]:
    for t in [15, 30]:
        p = monthly_payment(principal, r, t)
        ti = total_interest(principal, r, t)
        print(f"{r:>6.1%} {t:>5d}yr  ${p:>12,.2f}   ${ti:>14,.0f}   {ti/principal:>12.2f}x")

### Reading the Comparison Table

**Interpreting the output:** The comparison table reveals the double impact of interest rates and term length:

- Going from 4% to 8% roughly *doubles* total interest at any given term.
- Going from 15 years to 30 years roughly *doubles* total interest at any given rate.
- The combined effect at 8%/30yr: you pay 1.64x the principal in interest — nearly $656,000 on a $400,000 loan.

This is why even a 0.5% rate reduction can save tens of thousands of dollars over the life of a mortgage.

### A Useful Mental Model

Here is a quick way to estimate total interest without a calculator:

| Rate | 15-year multiplier | 30-year multiplier |
|------|-------------------:|-------------------:|
| 4% | ~0.33x principal | ~0.69x principal |
| 6% | ~0.52x principal | ~1.16x principal |
| 8% | ~0.72x principal | ~1.64x principal |

So for a $400,000 mortgage at 6%, you can quickly estimate: total interest is roughly $400,000 x 1.16 = $464,000 for a 30-year term, or $400,000 x 0.52 = $208,000 for a 15-year term. The difference — $256,000 — is the price of lower monthly payments.

> **Key Concept:** The relationship between rate, term, and total interest is nonlinear. Small changes in rate have outsized effects over long time horizons because of compounding. A 1% rate reduction on a 30-year $400,000 mortgage saves roughly $80,000-$100,000 in total interest.

### The Rate Sensitivity of Mortgage Costs

One of the most important insights from the comparison table is how sensitive total cost is to the interest rate. Let us quantify this sensitivity.

For a $400,000 mortgage over 30 years, a 1-percentage-point change in rate has the following effects:

| Rate Change | Monthly Payment Change | Total Interest Change | % Change in Total Interest |
|:-----------:|:---------------------:|:--------------------:|:--------------------------:|
| 4% to 5% | +$204/mo | +$73,500 | +27% |
| 5% to 6% | +$210/mo | +$75,700 | +22% |
| 6% to 7% | +$215/mo | +$77,500 | +19% |
| 7% to 8% | +$220/mo | +$78,900 | +16% |

Each additional percentage point costs roughly $75,000-$79,000 more in total interest. This is why mortgage rates are front-page news: a 0.5% change in the prevailing rate shifts hundreds of billions of dollars in aggregate mortgage costs across the economy.

> **CFA Exam Tip:** This rate sensitivity is analogous to the concept of **duration** in bond mathematics. A mortgage with a longer term has higher "duration" — its total cost is more sensitive to rate changes. This connection between mortgage sensitivity and bond duration becomes explicit in Section 9 when we discuss WAL.

### Practical Implications: Shopping for Rates

The comparison table makes a compelling case for rate shopping. If you can negotiate or shop your rate down by even 0.25%, the savings over 30 years are substantial:

- On a $400,000 / 30-year mortgage, 0.25% lower rate saves roughly $18,000-$20,000 in total interest
- This is equivalent to getting approximately 7-8 months of mortgage payments for free
- The effort of getting quotes from 3-5 lenders — perhaps a few hours of work — may be the highest-paid "hourly rate" you ever earn

> **Common Mistake:** Assuming the first rate quote is the best you can get. Mortgage rates vary by 0.25-0.50% or more across lenders on any given day. The Consumer Financial Protection Bureau (CFPB) estimates that failing to shop around costs the average borrower $1,500-$3,000 per year.

## 4. Amortization Schedule

### The Key Insight: Early Payments Are Mostly Interest

Each monthly payment is split between **interest** on the outstanding balance and **principal** repayment:

- Interest payment: $I_k = B_{k-1} \cdot r$ (interest on the current balance)
- Principal payment: $P_k = PMT - I_k$ (whatever is left after interest)
- New balance: $B_k = B_{k-1} - P_k$

These three equations, applied recursively, generate the entire amortization schedule. They are deceptively simple, but the pattern they produce surprises most people.

### Why This Split Matters Enormously

Let us walk through the first and 300th payments of our $400,000 mortgage at 6.5% (monthly payment = $2,528.27) in detail.

**Month 1:**
- Balance: $400,000.00
- Interest: $400,000 \times 0.065/12 = \$2,166.67$
- Principal: $2,528.27 - 2,166.67 = \$361.60$
- New balance: $400,000 - 361.60 = \$399,638.40$

That is right — in the first month, **85.7% of your payment goes to interest** and only 14.3% reduces your loan balance. After your first payment of $2,528, you still owe $399,638. It feels like you are treading water.

**Month 300 (Year 25):**
- Balance: ~$117,000
- Interest: $117,000 \times 0.065/12 = \$634$
- Principal: $2,528 - 634 = \$1,894$
- New balance: ~$115,106

Now **74.9% goes to principal** and only 25.1% to interest. The balance is dropping rapidly.

> **Key Concept:** The interest/principal split is not a bank trick — it is a mathematical consequence of charging interest on the *outstanding* balance. When the balance is large (early years), interest is large and principal is small. When the balance is small (late years), the reverse is true. The monthly payment never changes; only the split shifts.

### The Crossover Point

There is a specific month where the split reaches 50/50 — half to interest, half to principal. For our $400,000 mortgage at 6.5%:
- The crossover happens around **year 19-20** of a 30-year mortgage.
- For the first 19+ years, you are paying more interest than principal each month.

This means that for nearly two-thirds of the mortgage term, the majority of each payment goes to the bank as interest, not to building your equity.

### What This Means for Real Estate Decisions

This front-loading of interest has profound implications:

1. **Selling early is expensive.** If you sell after 5 years, you have made 60 payments totaling ~$151,700, but your loan balance has only decreased by ~$25,000. The remaining ~$126,700 went to interest. Your equity is the down payment plus $25,000 of principal paydown (plus or minus home price changes).

2. **Equity builds slowly, then rapidly.** The balance curve is concave — nearly flat at first, then dropping steeply near the end.

3. **Refinancing resets the clock.** When you refinance, you start a new amortization schedule. If you refinance at year 10, your new loan is once again front-loaded with interest. This is a hidden cost of refinancing that many people overlook.

> **Common Mistake:** Many homeowners believe they are "building equity" quickly because they are making large monthly payments. In reality, most of those early payments are interest. True equity accumulation depends primarily on home price appreciation in the early years, not on principal paydown.

### The Balance Recursion Formula

We can also express the balance at month $k$ in closed form:

$$B_k = PV \cdot \frac{(1+r)^k \cdot [(1+r)^n - (1+r)^k]}{(1+r)^n - 1} \cdot \frac{1}{(1+r)^k}$$

Simplifying:

$$B_k = PV \cdot \frac{(1+r)^n - (1+r)^k}{(1+r)^n - 1}$$

This formula lets you compute the balance at any month directly, without iterating through all prior months. It is useful for refinancing calculations (where you need the remaining balance at month 60, for example) and for verifying amortization tables.

### The Closed-Form Balance Formula

While the recursive approach (subtract principal from balance each month) is intuitive, there is also a closed-form expression for the balance at any month $k$:

$$B_k = PV \cdot \frac{(1+r)^n - (1+r)^k}{(1+r)^n - 1}$$

Let us verify this makes sense at the boundaries:
- At $k = 0$: $B_0 = PV \cdot \frac{(1+r)^n - 1}{(1+r)^n - 1} = PV$ (the full principal, correct)
- At $k = n$: $B_n = PV \cdot \frac{(1+r)^n - (1+r)^n}{(1+r)^n - 1} = 0$ (fully paid off, correct)

Similarly, the interest and principal portions of payment $k$ have closed forms:

$$I_k = PMT \cdot \left[1 - \frac{(1+r)^{k-1}}{(1+r)^n - 1}\cdot r \cdot n + ... \right]$$

In practice, the recursive approach is simpler to implement and understand. But the closed-form balance formula is useful when you need the balance at a specific month (e.g., for refinancing analysis in Section 8) without computing the entire schedule.

> **Key Concept:** The balance formula $B_k = PV \cdot \frac{(1+r)^n - (1+r)^k}{(1+r)^n - 1}$ shows that the balance is a function of the ratio of two exponentials. Early on, $(1+r)^k$ is much smaller than $(1+r)^n$, so the numerator is close to $(1+r)^n$ and $B_k \approx PV$. The balance only drops significantly when $(1+r)^k$ becomes a meaningful fraction of $(1+r)^n$, which happens in the later years.

### Deriving the Interest and Principal Portions

We can also derive closed-form expressions for the interest and principal portions of any payment $k$.

The interest portion of payment $k$ is:
$$I_k = B_{k-1} \cdot r = PV \cdot r \cdot \frac{(1+r)^n - (1+r)^{k-1}}{(1+r)^n - 1}$$

The principal portion is:
$$P_k = PMT - I_k = PV \cdot \frac{r \cdot (1+r)^{k-1}}{(1+r)^n - 1}$$

Notice that $P_k$ grows geometrically: $P_{k+1} = P_k \cdot (1+r)$. Each month, the principal portion is exactly $(1+r)$ times the previous month's principal portion. This means principal payments grow by about 0.54% per month (for a 6.5% annual rate) — a steady exponential increase that eventually dominates the payment.

This geometric growth of the principal portion is the mathematical engine behind the "slow start, fast finish" character of amortization.

### Implementation

The code below generates the full amortization schedule and displays the first and last few months. The `amortization_schedule` function also supports an optional `extra_payment` parameter that we will use in Section 6.

Pay attention to the first and last rows of the output — they illustrate the dramatic shift from interest-dominated to principal-dominated payments.

**What each column means:**
- **Month:** The payment number (1 through 360 for a 30-year mortgage)
- **Payment:** Your fixed monthly payment (constant for a fixed-rate mortgage, plus any extra payment)
- **Principal:** The portion of your payment that reduces the loan balance — this is the part that builds your equity
- **Interest:** The portion that goes to the lender as compensation for lending you money — this is the "cost" of borrowing
- **Balance:** The remaining amount you owe after the payment is applied

The first few months show the painful reality of early amortization; the last few months show the satisfying acceleration of principal paydown.

In [ ]:
def amortization_schedule(principal, annual_rate, years, extra_payment=0.0):
    """Generate full amortization schedule.
    
    Returns
    -------
    dict with keys: month, payment, principal_paid, interest_paid, 
                    extra_paid, balance, cumulative_interest
    """
    r = annual_rate / 12.0
    n = years * 12
    pmt = monthly_payment(principal, annual_rate, years)
    
    months = []
    payments = []
    principals = []
    interests = []
    extras = []
    balances = []
    cum_interest = []
    
    balance = principal
    total_interest_paid = 0.0
    
    for month in range(1, n + 1):
        if balance <= 0:
            break
        
        interest = balance * r
        principal_part = pmt - interest
        extra = min(extra_payment, balance - principal_part)  # don't overpay
        extra = max(extra, 0)
        
        balance -= (principal_part + extra)
        if balance < 0:
            # Adjust last payment
            principal_part += balance
            balance = 0.0
        
        total_interest_paid += interest
        
        months.append(month)
        payments.append(pmt + extra)
        principals.append(principal_part + extra)
        interests.append(interest)
        extras.append(extra)
        balances.append(max(balance, 0))
        cum_interest.append(total_interest_paid)
    
    return {
        'month': np.array(months),
        'payment': np.array(payments),
        'principal_paid': np.array(principals),
        'interest_paid': np.array(interests),
        'extra_paid': np.array(extras),
        'balance': np.array(balances),
        'cumulative_interest': np.array(cum_interest),
    }


# --- Generate schedule ---
sched = amortization_schedule(400_000, 0.065, 30)

# Display first and last few months
print(f"{'Month':>6s} {'Payment':>10s} {'Principal':>12s} {'Interest':>10s} {'Balance':>14s}")
print("-" * 60)
for i in list(range(5)) + ['...'] + list(range(-5, 0)):
    if i == '...':
        print(f"{'...':>6s}")
        continue
    print(f"{sched['month'][i]:>6d} ${sched['payment'][i]:>9,.2f} "
          f"${sched['principal_paid'][i]:>10,.2f} ${sched['interest_paid'][i]:>9,.2f} "
          f"${sched['balance'][i]:>12,.2f}")

### Reading the Amortization Table

**Interpreting the output:** Compare the first and last rows carefully:

| | Month 1 | Month 360 |
|---|---------|-----------|
| Interest | ~$2,167 (85.7% of payment) | ~$14 (0.5% of payment) |
| Principal | ~$362 (14.3% of payment) | ~$2,514 (99.5% of payment) |
| Balance change | Barely moves | Drops to zero |

The balance barely budges in the early years but drops rapidly near the end. This is the fundamental asymmetry of amortization.

### A Concrete Way to Think About It

Imagine your mortgage payment as a pie chart that shifts every month:

- **Month 1:** A huge coral slice (interest) and a tiny blue sliver (principal)
- **Month 180 (year 15):** The slices are getting closer to equal
- **Month 300 (year 25):** A huge blue slice (principal) and a tiny coral sliver (interest)
- **Month 360 (year 30):** Almost entirely blue — your last payment is nearly all principal

The visualizations below make these patterns visually clear.

> **CFA Exam Tip:** Amortization schedules appear in the CFA curriculum for both mortgage analysis and bond amortization (premium/discount bonds). The mechanics are identical: a fixed cash flow split between an interest component (based on carrying value) and a principal component (the remainder).

### Equity Accumulation: A Closer Look

Your home equity from mortgage payments alone (ignoring home price changes) is simply:

$$\text{Equity}_k = PV - B_k = PV \cdot \frac{(1+r)^k - 1}{(1+r)^n - 1}$$

For our $400,000 mortgage at 6.5%:

| Year | Equity from Payments | % of Principal | Cumulative Interest Paid |
|:----:|:-------------------:|:--------------:|:------------------------:|
| 1 | ~$4,500 | 1.1% | ~$25,800 |
| 3 | ~$14,000 | 3.5% | ~$76,500 |
| 5 | ~$25,000 | 6.3% | ~$126,000 |
| 10 | ~$60,000 | 15.0% | ~$243,000 |
| 15 | ~$112,000 | 28.0% | ~$343,000 |
| 20 | ~$190,000 | 47.5% | ~$417,000 |
| 25 | ~$300,000 | 75.0% | ~$470,000 |
| 30 | $400,000 | 100.0% | ~$510,000 |

After 10 years and $303,000 in total payments, your equity from principal paydown is only $60,000 (15% of the loan). The other $243,000 went to the bank as interest.

This is why home price appreciation is so important for building wealth through homeownership in the early years. If your home appreciates 3% per year, after 5 years you have gained ~$63,000 in price appreciation versus only ~$25,000 in principal paydown. The price appreciation dominates.

> **Common Mistake:** Confusing total payments with equity. After 5 years, you have paid $151,700 but only built $25,000 in equity from principal paydown. If someone asks "how much equity do you have?", the answer depends on home price changes, not just how many payments you have made.

## 5. Amortization Visualization

Visualizing the amortization schedule reveals patterns that are hard to see in a table of numbers. We create four complementary views:

1. **Stacked area chart** — showing how each payment splits between principal and interest over time. This is the "shifting pie chart" animated across all 360 months.
2. **Outstanding balance curve** — the familiar concave decay of the loan balance. Notice how the curve is nearly flat in the early years (you are barely paying down principal) and steep near the end.
3. **Cumulative totals** — when cumulative principal finally overtakes cumulative interest. This crossover happens surprisingly late.
4. **Interest share** — tracking the crossover from interest-dominant to principal-dominant payments, with the exact crossover year marked.

> **Key Concept:** The four visualizations below answer four distinct questions: (1) Where does each dollar of my payment go? (2) How fast is my balance shrinking? (3) Am I paying more interest or principal in total so far? (4) When does the interest/principal split reach 50/50? Together, they provide a complete visual picture of amortization dynamics.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

months_yr = sched['month'] / 12.0

# Top-left: stacked area - principal vs interest per payment
ax = axes[0, 0]
ax.fill_between(months_yr, 0, sched['interest_paid'], alpha=0.7, color=SECONDARY, label='Interest')
ax.fill_between(months_yr, sched['interest_paid'],
                sched['interest_paid'] + sched['principal_paid'],
                alpha=0.7, color=PRIMARY, label='Principal')
ax.set_xlabel('Year')
ax.set_ylabel('Monthly Payment ($)')
ax.set_title('Payment Breakdown Over Time')
ax.legend()

# Top-right: outstanding balance
ax = axes[0, 1]
ax.plot(months_yr, sched['balance'], color=PRIMARY, linewidth=2)
ax.fill_between(months_yr, sched['balance'], alpha=0.2, color=PRIMARY)
ax.set_xlabel('Year')
ax.set_ylabel('Outstanding Balance ($)')
ax.set_title('Outstanding Balance')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Bottom-left: cumulative interest vs principal
ax = axes[1, 0]
cum_principal = np.cumsum(sched['principal_paid'])
ax.plot(months_yr, sched['cumulative_interest'], color=SECONDARY, linewidth=2, label='Cumulative Interest')
ax.plot(months_yr, cum_principal, color=PRIMARY, linewidth=2, label='Cumulative Principal')
ax.set_xlabel('Year')
ax.set_ylabel('Cumulative Amount ($)')
ax.set_title('Cumulative Interest vs Principal')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Bottom-right: interest share of each payment
ax = axes[1, 1]
interest_share = sched['interest_paid'] / sched['payment'] * 100
ax.plot(months_yr, interest_share, color=SECONDARY, linewidth=2)
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
# Find crossover point
crossover_idx = np.where(interest_share < 50)[0]
if len(crossover_idx) > 0:
    crossover_yr = months_yr[crossover_idx[0]]
    ax.axvline(x=crossover_yr, color=TERTIARY, linestyle='--', alpha=0.7,
               label=f'50/50 crossover: year {crossover_yr:.1f}')
ax.set_xlabel('Year')
ax.set_ylabel('Interest Share of Payment (%)')
ax.set_title('Interest as Percentage of Payment')
ax.legend()

plt.suptitle('$400,000 Mortgage at 6.5% for 30 Years', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Interpreting the Four Panels

**Interpreting the output:**

- **Top-left (Payment Breakdown):** The coral (interest) and blue (principal) areas clearly show the gradual shift. Early years are dominated by interest; late years by principal. The total height of the stacked area is constant — your payment never changes.
- **Top-right (Outstanding Balance):** The balance drops slowly at first (convex region), then accelerates in later years. This concave shape is characteristic of all amortizing loans.
- **Bottom-left (Cumulative):** Notice how cumulative interest races ahead initially. The crossover where cumulative principal equals cumulative interest does not happen until late in the mortgage — typically around year 22-24 for a 30-year loan at 6.5%.
- **Bottom-right (Interest Share):** The crossover point where interest drops below 50% of each payment is marked. Before this point, more than half of every payment goes to the bank as interest.

### What This Means for Homeowners

If you sell your home after 5 years of a 30-year mortgage, you have made 60 payments totaling about $151,700, but your principal has only decreased by about $25,000. The remaining $126,700 went to interest. This is why building equity through mortgage payments alone is painfully slow in the early years — and why prepayment strategies can be so valuable.

Let us put this in an even starker way:

| Years Owned | Payments Made | Principal Paid | Interest Paid | Equity from Payments |
|:-----------:|:------------:|:--------------:|:-------------:|:-------------------:|
| 5 | $151,697 | ~$25,000 | ~$126,700 | 6.3% of loan |
| 10 | $303,394 | ~$60,000 | ~$243,400 | 15.0% of loan |
| 15 | $455,091 | ~$112,000 | ~$343,100 | 28.0% of loan |
| 20 | $606,789 | ~$190,000 | ~$416,800 | 47.5% of loan |
| 30 | $910,177 | $400,000 | ~$510,200 | 100% of loan |

After 15 years — half the mortgage term — you have only paid down 28% of the principal. The remaining 72% is paid in the last 15 years when the interest burden is lighter.

> **Key Concept:** The concave shape of the balance curve means that equity builds *non-linearly*. The first half of the mortgage term builds much less equity than the second half. This has important implications for decisions about when to sell, refinance, or prepay.

### The Mathematics of the Balance Curve

Why does the balance curve have that distinctive concave shape? Mathematically, the balance at month $k$ is:

$$B_k = PV \cdot \frac{(1+r)^n - (1+r)^k}{(1+r)^n - 1}$$

The second derivative of this with respect to $k$ is positive (convex in the "amount remaining" sense), meaning the *rate of decrease* accelerates over time. In the first year, the balance decreases by about $4,500. In the last year, it decreases by about $30,000. The acceleration is exponential, driven by the $(1+r)^k$ term.

### Connecting to Bond Mathematics

If you are familiar with bond pricing, the amortization schedule has a direct parallel. Consider a bond priced at par with a coupon rate equal to the yield:

- **Bond coupon** corresponds to **mortgage payment** (the fixed cash flow)
- **Bond yield times face value** corresponds to **interest portion** of the mortgage payment
- **Bond amortization of premium/discount** corresponds to **principal portion** of the mortgage payment

The key difference: a bond pays its entire principal at maturity (bullet payment), while a mortgage amortizes principal over the entire life. This difference is what makes mortgage duration shorter than bond duration for the same maturity.

> **CFA Exam Tip:** Understanding the shape of the amortization schedule helps with MBS duration and convexity calculations. Because principal payments are back-loaded (more principal in later years), the duration of a mortgage pool is affected differently by prepayments depending on when they occur. Early prepayments shorten duration more than late prepayments.

## 6. Prepayment Analysis

### The Power of Extra Payments

Extra monthly payments go **entirely to principal reduction** (there is no additional interest on the extra amount). This has a compounding benefit: every dollar of extra principal today saves you interest on that dollar for the remaining life of the loan.

Think of it this way: if you pay an extra $500 in month 1, that $500 no longer accrues interest for the remaining 359 months. At 6.5%, that single $500 extra payment saves you:

$$\$500 \times \left[(1 + 0.065/12)^{359} - 1\right] \approx \$500 \times 5.97 \approx \$2,985$$

A single $500 extra payment in month 1 saves nearly $3,000 in interest over the life of the loan. That is a 6x return, guaranteed.

### The NPV Reasoning

Think of each dollar of extra prepayment as an investment that earns your mortgage interest rate (6.5% in our example) risk-free and tax-adjusted. Where else can you get a guaranteed 6.5% return? For most people, prepaying a mortgage is one of the best low-risk investments available.

However, there are reasons *not* to prepay:
- If you have higher-rate debt (credit cards at 20%), pay that first.
- If your mortgage rate is very low (e.g., 3% from 2020-2021), investing in the stock market historically returns more.
- If you need the liquidity for emergencies.
- If you itemize taxes and the mortgage interest deduction is valuable to you.

The decision framework is simple: **prepay if your mortgage rate exceeds your after-tax expected return on alternative investments, adjusted for risk.**

### Why Early Extra Payments Save More Than Late Ones

This is crucial and often misunderstood. An extra $1,000 in year 1 has 29 years to compound its savings. An extra $1,000 in year 25 has only 5 years. The math is dramatic:

| When Extra $1,000 Is Paid | Approximate Interest Saved |
|:-------------------------:|:--------------------------:|
| Year 1 | ~$5,700 |
| Year 5 | ~$4,200 |
| Year 10 | ~$2,900 |
| Year 15 | ~$1,800 |
| Year 20 | ~$900 |
| Year 25 | ~$350 |

The savings from early prepayment are roughly **16 times larger** than from late prepayment. This is the power of compound interest working in your favor.

> **Key Concept:** Extra payments have the biggest impact when made *early* in the mortgage, because each dollar reduces the principal on which interest compounds for the remaining decades. A $1,000 extra payment in year 1 saves far more interest than a $1,000 extra payment in year 25.

> **Common Mistake:** Some people make extra payments in the final years of their mortgage thinking it will save a lot of interest. By that point, most of each payment is already going to principal and the remaining balance is small. The time to make extra payments is early — ideally in the first 5-10 years.

### Worked Example: $200/Month Extra on a $400,000 Mortgage

On our $400,000 / 6.5% / 30-year mortgage (base payment: $2,528/mo):

Adding just $200/month extra:
- Reduces the term by approximately 5-6 years
- Saves approximately $100,000+ in total interest
- Total extra outlay: $200 x ~288 months = ~$57,600 for $100,000+ savings

That is a return of roughly 1.7x on the extra payments — guaranteed.

### The Diminishing Returns of Larger Extra Payments

There is a diminishing marginal return to extra payments. The first $100/month extra saves more *per dollar* than the next $100. This is because the first $100 eliminates the most expensive (longest-duration) principal. Each subsequent dollar eliminates slightly less expensive principal.

However, in absolute terms, more is always better. The question is whether the marginal dollar is better spent on prepayment or on other investments.

### Implementation

The code below analyzes the impact of various extra payment amounts on total interest and loan duration. We compare extra payments from $0 to $2,000/month and visualize both the balance paydown curves and total interest saved.

Watch for two things in the output:
1. The **nonlinear relationship** between extra payment size and interest saved — the first $100/month extra saves more per dollar than the last $100 of a $2,000/month extra
2. How dramatically the **payoff timeline shrinks** even with modest extra payments — $500/month extra can turn a 30-year mortgage into a 20-year mortgage

The left chart shows balance trajectories, and the right chart shows cumulative interest saved. Together, they tell the full prepayment story.

In [ ]:
principal = 400_000
rate = 0.065
term = 30

base_pmt = monthly_payment(principal, rate, term)
base_sched = amortization_schedule(principal, rate, term, 0)
base_total_interest = base_sched['cumulative_interest'][-1]
base_months = len(base_sched['month'])

print("Prepayment Analysis")
print("=" * 75)
print(f"Base: ${principal:,.0f} at {rate:.2%}, {term}-yr, Payment = ${base_pmt:,.2f}/mo")
print(f"Base total interest: ${base_total_interest:,.0f} over {base_months} months ({base_months/12:.1f} years)")
print()
print(f"{'Extra/mo':>12s} {'Months':>8s} {'Years':>8s} {'Time Saved':>12s} {'Total Interest':>16s} {'Interest Saved':>16s}")
print("-" * 75)

extra_payments = [0, 100, 200, 500, 1000, 1500, 2000]
results = []

for extra in extra_payments:
    s = amortization_schedule(principal, rate, term, extra)
    n_months = len(s['month'])
    tot_int = s['cumulative_interest'][-1]
    months_saved = base_months - n_months
    int_saved = base_total_interest - tot_int
    results.append((extra, n_months, tot_int, months_saved, int_saved))
    print(f"  ${extra:>8,.0f}   {n_months:>6d}   {n_months/12:>6.1f}   {months_saved:>8d} mo    ${tot_int:>13,.0f}    ${int_saved:>13,.0f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
colors_prep = plt.cm.viridis(np.linspace(0, 0.8, len(extra_payments)))
for (extra, _, _, _, _), color in zip(results, colors_prep):
    s = amortization_schedule(principal, rate, term, extra)
    ax.plot(s['month'] / 12, s['balance'], color=color, linewidth=2,
            label=f'+${extra:,.0f}/mo')
ax.set_xlabel('Year')
ax.set_ylabel('Outstanding Balance ($)')
ax.set_title('Balance with Extra Payments')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

ax = axes[1]
extras_arr = [r[0] for r in results]
savings_arr = [r[4] for r in results]
ax.bar(range(len(extras_arr)), savings_arr, color=TERTIARY, alpha=0.8)
ax.set_xticks(range(len(extras_arr)))
ax.set_xticklabels([f'${e:,.0f}' for e in extras_arr], rotation=45)
ax.set_xlabel('Extra Monthly Payment')
ax.set_ylabel('Interest Saved ($)')
ax.set_title('Total Interest Saved')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

### Reading the Prepayment Results

**Interpreting the output:** The prepayment analysis reveals a powerful non-linear effect:

- $100/month extra saves roughly $60K-80K in interest
- $500/month extra saves roughly $170K-200K
- $1,000/month extra (roughly 40% increase in payment) saves roughly $240K-280K and cuts the term nearly in half

The balance paydown curves (left plot) show dramatically different trajectories. The bar chart (right) shows that interest savings increase rapidly with extra payment size, but with diminishing marginal returns.

### A Practical Strategy: The Biweekly Mortgage

One popular prepayment strategy is to make biweekly (every 2 weeks) half-payments instead of monthly payments. Since there are 26 biweekly periods per year (not 24), this is equivalent to making 13 monthly payments per year instead of 12 — an extra payment per year that can shave 5-7 years off a 30-year mortgage.

For our example: one extra monthly payment per year = $2,528. Spread over 12 months, that is an extra ~$211/month. According to our table, that is roughly equivalent to the $200/month extra scenario — saving approximately $100,000+ in interest and cutting 5-6 years off the term.

### When NOT to Prepay: The Opportunity Cost Argument

Suppose your mortgage rate is 3.5% (common for 2020-2021 originations). The long-run average stock market return is ~10% nominal, or ~7% real. Even after adjusting for risk and taxes:

$$\text{Expected stock return (after tax)} \approx 7\% > 3.5\% = \text{Mortgage rate}$$

In this case, investing the extra $200/month in a diversified index fund has a *higher expected return* than prepaying the mortgage. The prepayment is guaranteed; the stock return is not. But over long horizons (20+ years), the probability of stocks beating 3.5% is historically very high.

This is why financial advisors often say "do not rush to prepay a low-rate mortgage." The math supports this — *if* you actually invest the difference rather than spending it.

> **CFA Exam Tip:** The prepayment decision is fundamentally an NPV problem: compare the present value of interest savings from prepayment to the present value of returns from alternative investments. The discount rate should reflect your personal opportunity cost of capital.

### The Mathematical Proof: Why Early Prepayments Save More

Let us prove rigorously why an extra dollar in month $k$ saves more than in month $k+1$.

An extra dollar of principal in month $k$ eliminates interest charges on that dollar for the remaining $n - k$ months. The total interest saved is:

$$\text{Savings}(k) \approx \$1 \times \left[(1+r)^{n-k} - 1\right]$$

Since $(1+r)^{n-k}$ is a decreasing function of $k$ (the exponent shrinks as $k$ increases), the savings decrease monotonically with the month of prepayment.

For our mortgage ($r = 0.065/12$, $n = 360$):
- Extra $1 in month 1: saves $(1.00542)^{359} - 1 = \$5.97$
- Extra $1 in month 180: saves $(1.00542)^{180} - 1 = \$1.65$
- Extra $1 in month 300: saves $(1.00542)^{60} - 1 = \$0.38$

The dollar invested in month 1 saves **15.7 times more** than the dollar invested in month 300. This is the mathematical basis for the advice "prepay early, not late."

### Lump-Sum Prepayments vs Monthly Extra Payments

Some homeowners come into a windfall (bonus, inheritance, stock vesting) and wonder whether to make a lump-sum prepayment or invest the money and make monthly extras. The analysis depends on:

1. **Timing of lump sum:** A lump sum today is better than spreading the same total amount over future months (because the lump sum starts saving interest immediately on the full amount).

2. **Alternative return:** If you can invest at a rate higher than your mortgage rate, investing is mathematically better (though riskier).

3. **Behavioral factors:** Many people who plan to make monthly extras eventually stop. The lump sum has the advantage of being a one-time decision.

**Example:** $12,000 lump sum in month 1 vs $100/month extra for 120 months (same total).

The lump sum saves more interest because the full $12,000 starts reducing principal immediately, while the monthly approach only gradually builds up the prepaid amount.

> **Key Concept:** The time value of prepayment is captured by the formula $\text{Savings}(k) = (1+r)^{n-k} - 1$ per dollar prepaid in month $k$. This function decreases exponentially with $k$, quantifying exactly why early prepayment is so much more valuable than late prepayment.

## 7. Adjustable-Rate Mortgages (ARMs)

### The Fundamental Trade-off: Lower Initial Rate vs Rate Risk

An **adjustable-rate mortgage** (ARM) offers a lower initial rate than a comparable fixed-rate mortgage, but the rate resets periodically based on a market index. The most common structure is a **5/1 ARM**: fixed for 5 years, then adjusting annually.

Why would anyone take this risk? Because the initial rate is typically 0.5-1.5% below the comparable fixed rate. On a $400,000 mortgage, a 1% lower rate saves roughly $250/month during the fixed period — $15,000 over 5 years. The question is whether the risk of future rate increases is worth that upfront savings.

### Key ARM Components

| Component | Description | Example |
|-----------|-------------|---------|
| **Initial rate** | Below-market "teaser" rate during fixed period | 5.5% |
| **Index** | Market benchmark the rate tracks after fixed period | SOFR (Secured Overnight Financing Rate) |
| **Margin** | Fixed spread above the index | 1.75% |
| **Periodic cap** | Maximum rate change at each adjustment | 2% per year |
| **Lifetime cap** | Maximum total rate increase over the loan life | 5% (so max rate = 10.5%) |
| **Floor** | Minimum rate (usually the initial rate) | 5.5% |

### How the Rate Adjustment Works

After the fixed period ends, the new rate is calculated as:

$$\text{New rate} = \text{Index} + \text{Margin}$$

Subject to constraints:

$$\text{New rate} = \max\left(\text{Floor},\; \min\left(\text{Previous rate} + \text{Periodic cap},\; \text{Initial rate} + \text{Lifetime cap}\right)\right)$$

The caps are crucial — they prevent your rate from jumping to, say, 15% in a single year. But the lifetime cap still allows significant increases. In our example, the rate can reach up to $5.5\% + 5\% = 10.5\%$ over the loan's life.

### Three Rate Scenarios: When ARM Wins vs Loses

Let us think through three scenarios for our 5/1 ARM at 5.5% vs a fixed rate at 6.5%:

**Scenario 1: Rates Rise Sharply**
- After year 5, the index rises to 7%, making the ARM rate 8.75% (index + margin, subject to caps)
- ARM payments spike; total interest exceeds the fixed-rate mortgage
- **Fixed wins** by a wide margin

**Scenario 2: Rates Stay Flat**
- After year 5, the index stays around 4.75%, making the ARM rate ~6.5%
- ARM saves money during the initial period with lower rate, then matches the fixed
- **ARM wins** modestly (the 5-year savings are preserved)

**Scenario 3: Rates Fall**
- After year 5, the index drops to 3%, making the ARM rate ~4.75%
- ARM payments actually *decrease* after the fixed period
- **ARM wins** significantly

### When Do ARMs Make Sense?

| Scenario | ARM may be better | Fixed may be better |
|----------|------------------|-------------------|
| Plan to sell/move within 5-7 years | You benefit from low initial rate, sell before adjustments | N/A |
| Rates expected to fall | Adjustments may actually lower your rate | N/A |
| N/A | N/A | Plan to stay 10+ years |
| N/A | N/A | Rates expected to rise |
| N/A | N/A | You need payment predictability |

### The Payment Shock Problem

The biggest risk of ARMs is **payment shock** — a sudden jump in monthly payment when the rate adjusts. If SOFR rises to 7% and your margin is 1.75%, your rate could jump to 8.75% (subject to caps). On a $400,000 balance, that could mean $500-800 more per month.

This is exactly what happened during 2006-2008: millions of homeowners with ARMs saw their payments spike when rates reset, contributing to the foreclosure crisis. Many borrowers had qualified for the mortgage based on the teaser rate and could not afford the adjusted payments.

> **Key Concept:** An ARM is essentially a bet on future interest rates. If rates stay flat or decline, the ARM borrower wins (lower total interest). If rates rise significantly, the ARM borrower loses (higher payments, possibly much higher total interest). The caps provide a ceiling, but that ceiling can still be painfully high.

> **CFA Exam Tip:** The CFA curriculum covers ARMs in the context of MBS analysis. You should understand how rate caps affect the interest rate risk of ARM-backed MBS and why prepayment behavior differs between ARM and fixed-rate pools. ARM borrowers are more likely to refinance into fixed-rate mortgages when rates drop, creating asymmetric prepayment patterns.

### The ARM Payment Recalculation

An important detail: when the ARM rate adjusts, the payment is *recalculated* based on the remaining balance and remaining term at the new rate. This is different from just applying the new rate to the old payment. The formula is:

$$PMT_{\text{new}} = B_{\text{remaining}} \cdot \frac{r_{\text{new}}(1+r_{\text{new}})^{n_{\text{remaining}}}}{(1+r_{\text{new}})^{n_{\text{remaining}}} - 1}$$

This means the payment can change significantly at each adjustment, even with the same rate change, because the remaining balance and term are different.

### Implementation

The code below simulates a 5/1 ARM in a rising-then-falling rate scenario and compares it to a fixed-rate mortgage. The simulation includes:

- Realistic rate adjustments with periodic and lifetime caps
- Payment recalculation at each adjustment date using the annuity formula on the remaining balance and term
- Three-panel comparison: rate path, payment trajectory, and cumulative interest

The index rates are chosen to represent a realistic economic cycle: rates rise for several years after the fixed period ends, peak around year 12, then gradually decline back to lower levels. This tests both the upside risk and eventual recovery of ARM performance.

**Reading the three panels:**
- **Left panel (Rate Comparison):** Shows the ARM rate stepping up and down annually vs the flat fixed rate. The steps illustrate how caps limit annual adjustments.
- **Middle panel (Payment Shock):** Shows the monthly payment volatility of the ARM vs the stability of the fixed. The vertical jumps are the "payment shocks" that ARM borrowers experience.
- **Right panel (Cumulative Interest):** The ultimate scorecard — which mortgage costs more in total interest over the full 30 years.

In [ ]:
def arm_schedule(principal, initial_rate, term_years, fixed_years,
                 index_rates, margin, periodic_cap, lifetime_cap, floor=0.0):
    """Generate amortization schedule for an adjustable-rate mortgage.
    
    Parameters
    ----------
    principal : float
    initial_rate : float — annual rate during fixed period
    term_years : int — total loan term
    fixed_years : int — years at initial fixed rate
    index_rates : array — annual index rates for each adjustment period
    margin : float — spread above index
    periodic_cap : float — max annual rate change at reset
    lifetime_cap : float — max total rate increase from initial rate
    floor : float — minimum rate
    """
    n_total = term_years * 12
    balance = principal
    current_rate = initial_rate
    
    months = []
    payments = []
    interests = []
    principals_paid = []
    balances = []
    rates = []
    
    for month in range(1, n_total + 1):
        if balance <= 0:
            break
        
        year = (month - 1) // 12
        
        # Rate adjustment after fixed period
        if year >= fixed_years and (month - 1) % 12 == 0:
            adj_idx = min(year - fixed_years, len(index_rates) - 1)
            target_rate = index_rates[adj_idx] + margin
            
            # Apply caps
            max_rate = min(current_rate + periodic_cap,
                          initial_rate + lifetime_cap)
            min_rate = max(current_rate - periodic_cap, floor)
            current_rate = np.clip(target_rate, min_rate, max_rate)
        
        # Recalculate payment based on remaining balance and term
        remaining_months = n_total - month + 1
        r_monthly = current_rate / 12.0
        
        if abs(r_monthly) < 1e-15:
            pmt = balance / remaining_months
        else:
            pmt = balance * r_monthly * (1 + r_monthly) ** remaining_months / \
                  ((1 + r_monthly) ** remaining_months - 1)
        
        interest = balance * r_monthly
        principal_part = pmt - interest
        balance -= principal_part
        
        months.append(month)
        payments.append(pmt)
        interests.append(interest)
        principals_paid.append(principal_part)
        balances.append(max(balance, 0))
        rates.append(current_rate)
    
    return {
        'month': np.array(months),
        'payment': np.array(payments),
        'interest_paid': np.array(interests),
        'principal_paid': np.array(principals_paid),
        'balance': np.array(balances),
        'rate': np.array(rates),
    }


# --- 5/1 ARM example: rates rising scenario ---
principal = 400_000
initial_rate = 0.055  # 5.5% teaser
# Simulate index rates rising over time
index_rates = np.array([0.04, 0.045, 0.05, 0.055, 0.06, 0.065, 0.07, 0.07,
                        0.065, 0.06, 0.055, 0.05, 0.045, 0.04, 0.04, 0.04,
                        0.04, 0.04, 0.04, 0.04, 0.04, 0.04, 0.04, 0.04, 0.04])

arm = arm_schedule(principal, initial_rate, 30, 5, index_rates,
                   margin=0.0175, periodic_cap=0.02, lifetime_cap=0.05)

# Compare with fixed rate
fixed_rate = 0.065
fixed = amortization_schedule(principal, fixed_rate, 30)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
ax.plot(arm['month'] / 12, arm['rate'] * 100, color=SECONDARY, linewidth=2, label='ARM rate')
ax.axhline(y=fixed_rate * 100, color=PRIMARY, linestyle='--', linewidth=2, label=f'Fixed {fixed_rate:.1%}')
ax.axvline(x=5, color='gray', linestyle=':', alpha=0.5, label='End of fixed period')
ax.set_xlabel('Year')
ax.set_ylabel('Interest Rate (%)')
ax.set_title('Rate Comparison')
ax.legend(fontsize=9)

ax = axes[1]
ax.plot(arm['month'] / 12, arm['payment'], color=SECONDARY, linewidth=2, label='ARM')
ax.plot(fixed['month'] / 12, fixed['payment'], color=PRIMARY, linewidth=2, label='Fixed')
ax.set_xlabel('Year')
ax.set_ylabel('Monthly Payment ($)')
ax.set_title('Payment Shock Analysis')
ax.legend()

ax = axes[2]
arm_cum_int = np.cumsum(arm['interest_paid'])
ax.plot(arm['month'] / 12, arm_cum_int, color=SECONDARY, linewidth=2, label='ARM')
ax.plot(fixed['month'] / 12, fixed['cumulative_interest'], color=PRIMARY, linewidth=2, label='Fixed')
ax.set_xlabel('Year')
ax.set_ylabel('Cumulative Interest ($)')
ax.set_title('Cumulative Interest Comparison')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

print(f"ARM total interest:   ${arm_cum_int[-1]:>12,.0f}")
print(f"Fixed total interest: ${fixed['cumulative_interest'][-1]:>12,.0f}")

### Reading the ARM Comparison

**Interpreting the output:**

- **Left (Rate Comparison):** The ARM rate starts below the fixed rate during the 5-year fixed period, then rises above it when rates spike, and eventually comes back down. The caps limit the annual adjustment to 2%, preventing an immediate spike to the full indexed rate. Notice the stepped pattern — the rate changes only once per year.
- **Middle (Payment Shock):** The monthly payment jumps at each annual reset. In this scenario, the ARM payment exceeds the fixed payment for several years before falling back. The magnitude of the payment shock depends on how quickly rates rise and how binding the caps are.
- **Right (Cumulative Interest):** The ARM initially accumulates less interest (lower rate), but the rising-rate period causes it to catch up and potentially exceed the fixed-rate total.

### The Bottom Line on ARMs

In this particular scenario, the ARM still ends up paying slightly different total interest than the fixed. The key variables are: (1) how high and how long rates stay elevated, (2) whether you refinance or sell during the adjustment period, and (3) the cap structure.

### The Real-World ARM Strategy

In practice, many ARM borrowers do not hold the mortgage through the adjustment period. The strategy is often:

1. Take the ARM for the lower initial rate
2. Plan to refinance into a fixed rate *before* the first adjustment (or sell the house)
3. If rates have fallen by then, refinance into a lower fixed rate — best of both worlds
4. If rates have risen, you have been saving money during the initial period to partially offset higher future payments

This strategy works well in a stable or falling rate environment, but it failed spectacularly in 2006-2008 when home prices fell (making refinancing impossible) and rates reset higher simultaneously.

> **Common Mistake:** Taking an ARM solely because the initial payment is lower, without planning for the worst-case adjustment scenario. Always verify you can afford the maximum payment (initial rate + lifetime cap) before choosing an ARM.

### Scenario Analysis: Quantifying the ARM Gamble

To understand the ARM decision more concretely, consider three rate paths and their outcomes:

**Scenario A — Rates Rise 2% and Stay High:**
- ARM rate hits caps, stabilizes at ~7.5-8.5%
- ARM total interest exceeds fixed by $30,000-$60,000
- Monthly payment shock of $300-600/month during peak years
- Fixed-rate borrower sleeps soundly

**Scenario B — Rates Stay Flat:**
- ARM rate after fixed period is approximately equal to the fixed rate
- ARM saves ~$15,000 during the 5-year fixed period (lower initial rate)
- Total interest roughly comparable, slight ARM advantage
- ARM borrower takes some risk for modest reward

**Scenario C — Rates Fall 2%:**
- ARM rate drops to 3.5-4.5% after adjustments
- ARM saves $50,000-$80,000 in total interest
- Monthly payments actually decrease
- ARM borrower is rewarded handsomely for taking rate risk

### The Expected Value Calculation

If you assign probabilities to these scenarios:

| Scenario | Probability | ARM Advantage/Disadvantage |
|:--------:|:-----------:|:--------------------------:|
| Rates rise 2% | 30% | -$45,000 |
| Rates flat | 40% | +$15,000 |
| Rates fall 2% | 30% | +$65,000 |

Expected value of ARM vs fixed: $0.30 \times (-45{,}000) + 0.40 \times 15{,}000 + 0.30 \times 65{,}000 = +\$12{,}500$

Under these assumptions, the ARM has a positive expected value. But the 30% chance of being $45,000 worse off (and experiencing payment shock) may not be worth it for risk-averse borrowers. This is a classic risk-return trade-off.

> **Key Concept:** The ARM vs fixed decision is not about which product is "better" in an absolute sense. It is about your risk tolerance, time horizon, and view on future rates. Financial theory says you should only take the ARM risk if you are adequately compensated — meaning the initial rate discount is large enough relative to the risk of rate increases.

## 8. Refinancing Decision

### The Core Question

You have a mortgage at 7% and new rates are 5.5%. Should you refinance? The answer depends on a careful NPV analysis, not just the rate difference.

Refinancing is essentially an investment: you spend money today (closing costs) to save money in the future (lower monthly payments). Like any investment, it should be evaluated on its net present value.

### What Is Refinancing, Exactly?

Refinancing replaces your existing mortgage with a new one. Here is what happens:

1. You take out a new loan (at the lower rate) for the remaining balance
2. The new loan pays off the old loan completely
3. You begin making payments on the new loan
4. You pay closing costs (typically $5,000-$15,000) for the new loan

The result: lower monthly payments, but you have paid upfront costs and potentially extended your amortization schedule.

### The Refinancing Decision Framework

The decision involves weighing four factors:

1. **Monthly savings**: The difference between old and new payments
2. **Upfront costs**: Closing costs, appraisal, title insurance, etc. (typically $5,000-$15,000)
3. **Breakeven period**: How many months until cumulative savings exceed closing costs
4. **NPV**: The total present value of savings minus costs

### The NPV Approach (The Right Way)

The correct decision metric is net present value:

$$\text{NPV} = \sum_{k=1}^{N} \frac{\text{PMT}_{\text{old}} - \text{PMT}_{\text{new}}}{(1+d)^k} - \text{Closing Costs}$$

where $d$ is the monthly discount rate (typically the new mortgage rate divided by 12, or your personal opportunity cost).

**If NPV > 0, refinancing creates value. If NPV < 0, do not refinance.**

The NPV approach is superior because it accounts for the **time value** of the monthly savings stream. A $300/month savings for 25 years is not simply $300 x 300 = $90,000 in value — its present value is lower because future dollars are worth less than today's dollars.

### The Breakeven Approach (The Simple Way)

$$\text{Breakeven (months)} = \frac{\text{Closing Costs}}{\text{Monthly Savings}}$$

If you plan to stay in the house longer than the breakeven period, refinancing is generally worthwhile. This is a quick screening tool, but it ignores the time value of money.

### Worked Example: Should You Refinance from 7% to 5.5%?

**Current mortgage:** $400,000 at 7%, 30 years, 5 years in (60 months paid).
**New rate:** 5.5%, 25-year term. **Closing costs:** $8,000.

**Step 1 — Find remaining balance after 60 months:**

Using the closed-form balance formula:
$$B_{60} = 400{,}000 \times \frac{(1.00583)^{360} - (1.00583)^{60}}{(1.00583)^{360} - 1} \approx \$381{,}000$$

**Step 2 — Compute old and new payments:**
- Old payment: $PMT = 400{,}000 \times \frac{0.00583 \times (1.00583)^{360}}{(1.00583)^{360} - 1} \approx \$2{,}661$/mo
- New payment on $381,000 at 5.5% for 25 years: $\approx \$2{,}336$/mo
- Monthly savings: $\approx \$325$/mo

**Step 3 — Simple breakeven:**
$$\frac{\$8{,}000}{\$325/\text{mo}} \approx 25 \text{ months} \approx 2 \text{ years}$$

**Step 4 — NPV calculation:**
The present value of $325/month for 25 years (300 months) at a 5.5% discount rate minus $8,000 in closing costs gives a strongly positive NPV.

**Decision:** If you plan to stay more than 2 years, refinancing is clearly worthwhile. The NPV confirms this.

> **Key Concept:** The breakeven period is a quick screening tool, but NPV is the correct decision metric. The breakeven approach ignores the time value of money and the fact that savings continue to accumulate long after breakeven.

> **Common Mistake:** People sometimes refinance purely to lower their monthly payment without considering closing costs, the reset of amortization (starting over with interest-heavy payments), or their planned time horizon. Always run the full NPV calculation.

### The Hidden Cost: Amortization Reset

One subtlety that many borrowers miss: refinancing restarts your amortization schedule. After 5 years of your original 30-year mortgage, you have finally started paying down principal more aggressively. Refinancing into a new 25- or 30-year mortgage puts you back at the beginning, where most of each payment is interest again.

This is why it sometimes makes sense to refinance into a *shorter* term (e.g., from a 30-year at 7% to a 15-year at 5%), even if the monthly payment does not decrease much. You avoid the amortization reset penalty while locking in the lower rate.

### Implementation

The code below performs a complete refinancing analysis and shows how NPV varies with the rate reduction. The analysis includes:

- Remaining balance calculation (using the amortization schedule from Section 4)
- Old vs new payment comparison
- Simple breakeven period (closing costs divided by monthly savings)
- Full NPV calculation with monthly discounting
- Total interest savings accounting for closing costs
- Sensitivity analysis: how NPV changes across rate reductions from 0.1% to 3.0%

The sensitivity chart is particularly useful — it shows the minimum rate reduction needed to justify refinancing given the closing costs. The green-shaded region represents profitable refinancing; the coral-shaded region represents unprofitable refinancing. The crossover point is the minimum rate reduction at which NPV equals zero.

In [ ]:
def refinancing_analysis(original_principal, original_rate, original_term,
                          months_elapsed, new_rate, new_term,
                          closing_costs, discount_rate=None):
    """Analyze a refinancing decision.
    
    Parameters
    ----------
    original_principal : float
    original_rate, new_rate : float — annual rates
    original_term, new_term : int — years
    months_elapsed : int — months already paid on original
    closing_costs : float
    discount_rate : float — for NPV; if None, uses new_rate
    """
    if discount_rate is None:
        discount_rate = new_rate
    
    # Current outstanding balance
    old_pmt = monthly_payment(original_principal, original_rate, original_term)
    old_sched = amortization_schedule(original_principal, original_rate, original_term)
    remaining_balance = old_sched['balance'][months_elapsed - 1]
    remaining_months_old = original_term * 12 - months_elapsed
    
    # New mortgage
    new_principal = remaining_balance + closing_costs  # roll costs into loan
    new_pmt = monthly_payment(remaining_balance, new_rate, new_term)
    
    monthly_savings = old_pmt - new_pmt
    
    # Breakeven period
    if monthly_savings > 0:
        breakeven_months = int(np.ceil(closing_costs / monthly_savings))
    else:
        breakeven_months = np.inf
    
    # NPV of savings
    d_monthly = discount_rate / 12
    n_compare = min(remaining_months_old, new_term * 12)
    
    # Remaining interest on old loan
    old_remaining_interest = np.sum(old_sched['interest_paid'][months_elapsed:])
    
    # Total interest on new loan
    new_sched = amortization_schedule(remaining_balance, new_rate, new_term)
    new_total_interest = new_sched['cumulative_interest'][-1]
    
    # NPV of monthly savings
    months_arr = np.arange(1, n_compare + 1)
    npv = np.sum(monthly_savings / (1 + d_monthly) ** months_arr) - closing_costs
    
    return {
        'remaining_balance': remaining_balance,
        'old_payment': old_pmt,
        'new_payment': new_pmt,
        'monthly_savings': monthly_savings,
        'breakeven_months': breakeven_months,
        'npv': npv,
        'old_remaining_interest': old_remaining_interest,
        'new_total_interest': new_total_interest,
        'interest_saved': old_remaining_interest - new_total_interest - closing_costs,
    }


# --- Example ---
result = refinancing_analysis(
    original_principal=400_000,
    original_rate=0.07,
    original_term=30,
    months_elapsed=60,  # 5 years in
    new_rate=0.055,
    new_term=25,
    closing_costs=8_000,
)

print("Refinancing Analysis")
print("=" * 50)
print(f"Outstanding balance:  ${result['remaining_balance']:>12,.2f}")
print(f"Old payment:          ${result['old_payment']:>12,.2f}/mo")
print(f"New payment:          ${result['new_payment']:>12,.2f}/mo")
print(f"Monthly savings:      ${result['monthly_savings']:>12,.2f}/mo")
print(f"Closing costs:        ${8_000:>12,.2f}")
print(f"Breakeven:            {result['breakeven_months']:>9d} months ({result['breakeven_months']/12:.1f} years)")
print(f"NPV of refinancing:   ${result['npv']:>12,.2f}")
print(f"Total interest saved: ${result['interest_saved']:>12,.2f}")
print(f"\nDecision: {'REFINANCE' if result['npv'] > 0 else 'DO NOT REFINANCE'}")

# --- Sensitivity: NPV vs rate reduction ---
rate_reductions = np.linspace(0.001, 0.03, 50)
npvs = []
for dr in rate_reductions:
    r = refinancing_analysis(400_000, 0.07, 30, 60, 0.07 - dr, 25, 8_000)
    npvs.append(r['npv'])

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(rate_reductions * 100, npvs, color=PRIMARY, linewidth=2)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.fill_between(rate_reductions * 100, npvs, 0,
                where=np.array(npvs) >= 0, alpha=0.3, color=TERTIARY)
ax.fill_between(rate_reductions * 100, npvs, 0,
                where=np.array(npvs) < 0, alpha=0.3, color=SECONDARY)
ax.set_xlabel('Rate Reduction (percentage points)')
ax.set_ylabel('NPV of Refinancing ($)')
ax.set_title('Refinancing NPV vs Rate Reduction (Closing Costs = $8,000)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

### Reading the Refinancing Analysis

**Interpreting the output:** The NPV sensitivity chart shows that very small rate reductions (under ~0.5%) do not justify the $8,000 in closing costs. But beyond about 0.75% reduction, the NPV becomes strongly positive and grows roughly linearly with the rate reduction.

The exact breakeven rate reduction depends on closing costs, remaining term, and time horizon. This is why the old "refinance if you can save 1%" rule of thumb is roughly correct but not always precise.

### The 1% Rule of Thumb — When It Works and When It Fails

The conventional wisdom says "refinance if you can get at least 1% lower." This rule works reasonably well when:
- Closing costs are $5,000-$10,000
- You plan to stay at least 5 years
- You are refinancing a 30-year into a 30-year

It fails when:
- Closing costs are unusually high (e.g., cash-out refinance with higher fees)
- You plan to move soon (breakeven may be longer than your stay)
- You are far into your existing mortgage (the amortization reset penalty is larger)
- You are refinancing into a different term length (comparing apples to oranges)

> **CFA Exam Tip:** Refinancing analysis is a standard NPV problem. The CFA curriculum emphasizes that the relevant comparison is the present value of savings vs the present value of costs. Be careful about the discount rate — it should reflect the borrower's opportunity cost, not necessarily the mortgage rate.

### Tax Considerations (Brief Note)

In the United States, mortgage interest is tax-deductible for homeowners who itemize deductions (up to $750,000 of mortgage debt for loans originated after 2017). This means:

- Your effective mortgage rate is lower than the stated rate: $r_{\text{effective}} = r \times (1 - \text{tax rate})$
- At a 24% marginal tax bracket, a 6.5% mortgage effectively costs $6.5\% \times 0.76 = 4.94\%$
- This makes the hurdle for prepayment and refinancing higher — you need a bigger rate reduction to justify the action

The tax benefit reduces as you pay down the mortgage (less interest to deduct), which is another reason early-year prepayment has a higher opportunity cost than late-year prepayment.

### Sensitivity Analysis: What Drives the Refinancing Decision?

The NPV of refinancing depends on several variables. Here is how sensitive the decision is to each one:

| Variable | Higher Value Effect | Typical Range |
|----------|:------------------:|:-------------:|
| Rate reduction | Strongly increases NPV | 0.25% - 2.0% |
| Remaining term | Increases NPV (more months of savings) | 10 - 28 years |
| Closing costs | Decreases NPV (higher upfront cost) | $3,000 - $15,000 |
| Time horizon (how long you stay) | Increases NPV up to a point | 2 - 30 years |
| Discount rate | Decreases NPV (future savings worth less) | 3% - 7% |

The most important variables are the **rate reduction** and the **time horizon**. If you are getting a 1.5% rate reduction and plan to stay 10+ years, refinancing is almost always worthwhile. If the reduction is only 0.5% and you might move in 3 years, it is a close call.

### The Serial Refinancer's Dilemma

Some homeowners refinance multiple times, especially during falling rate environments. Each refinance:
1. Captures a rate reduction (good)
2. Incurs new closing costs (bad)
3. Resets the amortization clock (bad — more interest-heavy payments)
4. May extend the total payoff date if refinancing into a new 30-year term (potentially bad)

The NPV framework handles all of these effects correctly. But the behavioral reality is that many serial refinancers end up extending their mortgage indefinitely — taking out a new 30-year mortgage every 5-7 years and never reaching the principal-heavy later years of amortization.

> **Common Mistake:** Refinancing into a new 30-year mortgage when you are already 10 years into your current one. This resets 10 years of amortization progress. Instead, consider refinancing into a 20-year mortgage to maintain your payoff timeline while capturing the lower rate.

### Cash-Out Refinancing: A Different Beast

Cash-out refinancing is when you borrow more than your remaining balance, taking the difference as cash. For example, if your balance is $300,000 and your home is worth $500,000, you might refinance into a $400,000 mortgage and take $100,000 in cash.

This is not really a "refinancing" in the optimization sense — it is a new borrowing decision that happens to use your home as collateral. The NPV analysis is different: you are comparing the cost of this debt to the return on whatever you use the cash for (renovations, debt consolidation, investment).

Cash-out refinancing should be evaluated as a new loan, not compared to your existing mortgage.

## 9. Weighted Average Life & Prepayment Models

### From Personal Finance to Wall Street

Everything we have discussed so far is from the borrower's perspective. But mortgages do not just sit on bank balance sheets — they are packaged into **mortgage-backed securities (MBS)** and sold to investors. The MBS market is enormous: over $12 trillion in the United States alone.

For MBS investors, the key question is not "what is my monthly payment?" but rather **"how long, on average, until I get my money back?"** This is the weighted average life.

### Why Mortgage Investors Care About WAL

The **Weighted Average Life** (WAL) measures the average time until each dollar of principal is repaid:

$$\text{WAL} = \frac{\sum_{k=1}^{n} t_k \cdot P_k}{\sum_{k=1}^{n} P_k}$$

where $P_k$ is the principal repaid at time $t_k$ (in years).

**Intuition:** Think of WAL as the "center of gravity" of the principal repayment stream. If all principal were repaid in a single bullet payment at maturity (like a corporate bond), WAL would equal the maturity. But since mortgages amortize (and borrowers prepay), WAL is shorter — often much shorter — than the stated maturity.

### WAL vs Stated Maturity

For our $400,000 / 6.5% / 30-year mortgage:
- **Stated maturity:** 30 years
- **WAL (no prepayments):** ~21 years — because scheduled amortization alone returns principal gradually, with more coming later (back-loaded due to the interest/principal split)
- **WAL (100% PSA):** ~12-14 years — prepayments accelerate principal return
- **WAL (400% PSA):** ~4-5 years — aggressive prepayments return principal very quickly

The difference between 21 years and 5 years is enormous for investment planning. It affects which bonds the MBS competes with, how sensitive the price is to interest rates, and what return investors should expect.

### The PSA Prepayment Model

The Public Securities Association (PSA) benchmark models how quickly homeowners prepay their mortgages (through refinancing, selling, or extra payments).

The standard 100% PSA assumes a conditional prepayment rate (CPR) that:
1. **Ramps linearly** from 0% to 6% over the first 30 months ("seasoning ramp")
2. **Remains at 6%** thereafter (steady state)

$$\text{CPR}(t) = \min\left(\frac{t}{30}, 1\right) \times 6\%$$

### Why the Seasoning Ramp?

The ramp reflects empirical observation: borrowers who just took out a mortgage are unlikely to prepay immediately (they just went through the hassle of getting one). Over the first 2.5 years, prepayment rates gradually increase as:
- Some homeowners sell and move
- Some refinance as their financial situation changes
- Some begin making extra payments as incomes grow

After 30 months, the pool is "fully seasoned" and prepayment reaches its long-run rate.

### Converting CPR to SMM

The conditional prepayment rate (CPR) is annual. To use it in monthly calculations, we convert to the single monthly mortality (SMM):

$$\text{SMM} = 1 - (1 - \text{CPR})^{1/12}$$

This conversion accounts for compounding: if 6% of the pool prepays over a year, slightly less than 0.5% prepays each month (because the monthly prepayments compound).

### PSA Speeds Explained

| Speed | CPR at Month 30+ | Meaning | When It Happens |
|:-----:|:-----------------:|---------|-----------------|
| 0% PSA | 0% | No prepayments at all | Theoretical baseline |
| 100% PSA | 6% | Standard benchmark | "Normal" rate environment |
| 200% PSA | 12% | Twice the standard rate | Moderately falling rates |
| 400% PSA | 24% | Very fast prepayments | Aggressive refi wave (rates drop sharply) |

### Why Prepayment Speed Matters for MBS Investors

For MBS investors, prepayment speed determines both the WAL and the return on investment:

- **Faster prepayments** reduce WAL and return principal earlier. If you bought the MBS at a premium (above par), getting your money back sooner means less time to collect above-market coupons — you lose.
- **Slower prepayments** extend WAL and delay principal return. If you bought at a premium, you benefit from collecting above-market coupons longer.
- This asymmetry creates **negative convexity** for MBS — the investor faces unfavorable outcomes whether rates go up (extension risk) or down (prepayment risk).

### Extension Risk vs Prepayment Risk

| | Rates Fall | Rates Rise |
|---|-----------|------------|
| **What borrowers do** | Refinance (prepay faster) | Stay put (prepay slower) |
| **MBS investor impact** | Get principal back early, must reinvest at lower rates | Stuck in below-market coupons longer |
| **Risk name** | Prepayment risk (contraction risk) | Extension risk |

This "heads I lose, tails I lose" dynamic is the central challenge of MBS investing and the reason MBS trade at a yield spread above comparable Treasuries.

> **Key Concept:** Prepayment risk is the defining challenge of MBS investing. Unlike corporate bonds, where the borrower rarely prepays, mortgage borrowers routinely refinance when rates drop. This creates a unique risk profile that makes MBS analysis fundamentally different from corporate bond analysis.

> **CFA Exam Tip:** The CFA exam tests PSA prepayment concepts at Level II. You should understand the PSA ramp, how to convert between CPR and SMM, how prepayment speed affects WAL and MBS investor returns, and the concept of negative convexity in MBS. Practice computing SMM from CPR and vice versa — it is a common calculation question.

### Implementation

The code below computes WAL at different PSA speeds and visualizes the balance rundown. The `psa_prepayment_schedule` function implements the full PSA model:

1. Compute the CPR for each month using the seasoning ramp
2. Convert CPR to SMM
3. Apply scheduled amortization (from the standard annuity formula)
4. Apply prepayment: SMM times the remaining balance after scheduled principal
5. Track cumulative principal payments for WAL calculation

Watch how dramatically the balance curves diverge across PSA speeds, and how WAL drops from ~21 years (no prepayments) to under 5 years (400% PSA).

Note that at 0% PSA (no prepayments), the function reduces to a standard amortization schedule, and the WAL equals the principal-weighted average time of scheduled payments. At higher PSA speeds, the SMM factor removes an increasing fraction of the remaining balance each month, accelerating principal return and compressing the WAL.

The WAL vs PSA speed plot (right panel) reveals the nonlinear relationship: the first 100% of PSA speed reduces WAL dramatically (from ~21 to ~13 years), while going from 300% to 400% PSA only shaves off another year or two. This diminishing sensitivity at high speeds reflects the fact that most of the pool has already prepaid.

In [ ]:
def weighted_average_life(schedule):
    """Compute WAL from an amortization schedule."""
    times = schedule['month'] / 12.0  # in years
    principal_payments = schedule['principal_paid']
    return np.sum(times * principal_payments) / np.sum(principal_payments)


def psa_prepayment_schedule(principal, annual_rate, term_years, psa_speed=100):
    """Generate amortization schedule with PSA prepayment model.
    
    Parameters
    ----------
    psa_speed : float — PSA speed as percentage (100 = standard, 200 = 2x, etc.)
    """
    r = annual_rate / 12.0
    n = term_years * 12
    pmt = monthly_payment(principal, annual_rate, term_years)
    
    balance = principal
    months = []
    principal_payments = []
    interest_payments = []
    prepayments = []
    balances = []
    
    for month in range(1, n + 1):
        if balance <= 0.01:
            break
        
        # PSA CPR
        cpr = min(month / 30.0, 1.0) * 0.06 * (psa_speed / 100.0)
        smm = 1.0 - (1.0 - cpr) ** (1.0 / 12.0)
        
        interest = balance * r
        scheduled_principal = min(pmt - interest, balance)
        
        # Prepayment on remaining balance after scheduled principal
        remaining_after = balance - scheduled_principal
        prepayment = remaining_after * smm
        
        total_principal = scheduled_principal + prepayment
        balance -= total_principal
        
        months.append(month)
        principal_payments.append(total_principal)
        interest_payments.append(interest)
        prepayments.append(prepayment)
        balances.append(max(balance, 0))
    
    return {
        'month': np.array(months),
        'principal_paid': np.array(principal_payments),
        'interest_paid': np.array(interest_payments),
        'prepayment': np.array(prepayments),
        'balance': np.array(balances),
        'payment': np.array(interest_payments) + np.array(principal_payments),
    }


# --- Compare WAL at different PSA speeds ---
principal = 400_000
rate = 0.065
term = 30

psa_speeds = [0, 50, 100, 150, 200, 300, 400]
wals = []

print("Weighted Average Life vs PSA Speed")
print("-" * 45)
print(f"{'PSA Speed':>10s} {'WAL (years)':>14s} {'Actual Life':>14s}")
print("-" * 45)

for speed in psa_speeds:
    sched = psa_prepayment_schedule(principal, rate, term, speed)
    wal = weighted_average_life(sched)
    actual_life = len(sched['month']) / 12.0
    wals.append(wal)
    print(f"  {speed:>6d}%    {wal:>12.2f}      {actual_life:>10.1f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
colors_psa = plt.cm.plasma(np.linspace(0.1, 0.9, len(psa_speeds)))
for speed, color in zip([0, 100, 200, 400], [PRIMARY, SECONDARY, TERTIARY, ACCENT]):
    sched = psa_prepayment_schedule(principal, rate, term, speed)
    ax.plot(sched['month'] / 12, sched['balance'], linewidth=2,
            color=color, label=f'{speed}% PSA')
ax.set_xlabel('Year')
ax.set_ylabel('Outstanding Balance ($)')
ax.set_title('Balance Rundown at Different PSA Speeds')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

ax = axes[1]
ax.plot(psa_speeds, wals, 'o-', color=PRIMARY, linewidth=2, markersize=8)
ax.set_xlabel('PSA Speed (%)')
ax.set_ylabel('Weighted Average Life (years)')
ax.set_title('WAL vs Prepayment Speed')

plt.tight_layout()
plt.show()

### Reading the WAL Results

**Interpreting the output:**

- At 0% PSA (no prepayments), the WAL is about 21 years — well below the 30-year stated maturity because scheduled amortization alone returns principal gradually. The back-loading of principal (due to the interest/principal split) pushes WAL above the midpoint of 15 years.
- At 100% PSA (standard), WAL drops to about 12-14 years. This is the benchmark that MBS investors use for "normal" prepayment assumptions.
- At 400% PSA (aggressive prepayments), WAL can drop below 5 years. This represents a scenario like 2020-2021, when rates fell sharply and a massive refinancing wave hit the market.

The balance rundown curves show dramatically different profiles. At high PSA speeds, the mortgage pool is essentially paid off in 15-20 years even though the stated term is 30 years. This is why MBS investors must model prepayment speeds carefully — the effective life of the investment can vary enormously depending on the interest rate environment.

### What WAL Means in Practice

For an MBS investor holding a pool at 100% PSA with WAL of ~13 years:

- The investment behaves roughly like a 13-year bond for duration and interest rate sensitivity purposes
- But unlike a 13-year bond, the WAL can *change* if prepayment speeds change
- If rates drop and prepayments accelerate to 400% PSA, the WAL suddenly shrinks to ~5 years — the investor is forced to reinvest at lower rates
- If rates rise and prepayments slow to 50% PSA, the WAL extends to ~17 years — the investor is stuck in a below-market coupon

This variability of WAL is what makes MBS so challenging to manage in a portfolio context. It is also why MBS traders and risk managers spend enormous effort modeling prepayment speeds and their sensitivity to interest rates.

> **Key Concept:** WAL is not a fixed property of a mortgage pool — it depends on future prepayment behavior, which in turn depends on future interest rates. This rate-dependence of effective maturity is unique to amortizing securities and is the root cause of negative convexity in MBS.

### Connecting It All Together

The journey through this notebook traces a complete arc:

1. **Personal finance:** The annuity formula determines your payment. Amortization reveals why early payments are mostly interest. Prepayment and refinancing strategies can save hundreds of thousands of dollars.

2. **Institutional finance:** Those same mortgages, packaged into MBS, create complex securities whose behavior depends on millions of individual prepayment decisions. WAL and PSA models are the tools investors use to navigate this complexity.

Understanding both perspectives — the borrower and the investor — gives you a complete picture of mortgage mathematics.

### A Final Worked Example: Putting It All Together

Let us trace a single mortgage through all the concepts in this notebook to reinforce the connections.

**The Mortgage:** $300,000 at 6% for 30 years.

**Section 3 — Payment Calculation:**
$$r = 0.06/12 = 0.005, \quad n = 360$$
$$PMT = 300{,}000 \times \frac{0.005 \times (1.005)^{360}}{(1.005)^{360} - 1} = 300{,}000 \times \frac{0.005 \times 6.0226}{5.0226} = \$1{,}798.65$$

Total interest: $1{,}798.65 \times 360 - 300{,}000 = \$347{,}515$ (1.16x the principal).

**Section 4 — Month 1 vs Month 300:**
- Month 1: Interest = $300,000 \times 0.005 = \$1{,}500$ (83.4%), Principal = $\$299$ (16.6%)
- Month 300: Interest = $\$452$ (25.1%), Principal = $\$1{,}347$ (74.9%)

**Section 6 — Prepayment:**
Adding $200/month extra would save approximately $75,000 in interest and cut the term by about 5 years.

**Section 7 — ARM Alternative:**
A 5/1 ARM at 5% (vs 6% fixed) would save $~\$150$/month during the fixed period ($9,000 over 5 years), but faces rate risk after year 5.

**Section 8 — Refinancing:**
If after 5 years rates drop to 4.5%, the remaining balance is ~$279,000. Refinancing into a 25-year mortgage at 4.5% with $6,000 closing costs:
- Old payment: $1,799/mo. New payment: ~$1,552/mo. Savings: $247/mo.
- Breakeven: $6,000 / $247 = 24 months. NPV strongly positive if staying 3+ years.

**Section 9 — WAL (Investor Perspective):**
If this mortgage is in an MBS pool at 100% PSA, the WAL is approximately 12-13 years — meaning investors get their money back, on average, in about 13 years, not 30.

> **Key Concept:** Every concept in this notebook connects to every other. The annuity formula determines the payment; the payment split determines the amortization schedule; the amortization schedule determines prepayment savings, refinancing breakevens, and WAL. Mastering the core formula and the amortization mechanics gives you the foundation for all mortgage-related analysis.

### Summary of Key Formulas

| Formula | Purpose |
|---------|---------|
| $PMT = PV \cdot \frac{r(1+r)^n}{(1+r)^n - 1}$ | Monthly payment |
| $I_k = B_{k-1} \cdot r$ | Interest portion of payment $k$ |
| $P_k = PMT - I_k$ | Principal portion of payment $k$ |
| $B_k = PV \cdot \frac{(1+r)^n - (1+r)^k}{(1+r)^n - 1}$ | Balance after payment $k$ |
| $\text{NPV} = \sum \frac{\Delta PMT}{(1+d)^k} - C$ | Refinancing decision |
| $\text{WAL} = \frac{\sum t_k P_k}{\sum P_k}$ | Weighted average life |
| $\text{SMM} = 1 - (1-\text{CPR})^{1/12}$ | Monthly prepayment rate |
| $\text{CPR}(t) = \min(t/30, 1) \times 6\% \times \text{PSA}/100$ | PSA prepayment model |

## 10. References

### Textbooks

1. Fabozzi, F. J. *Fixed Income Mathematics*, 4th ed. McGraw-Hill, 2006. — Chapters on mortgage mathematics and amortization. The definitive quantitative reference for fixed-income professionals.
2. Fabozzi, F. J. *The Handbook of Mortgage-Backed Securities*, 7th ed. Oxford University Press, 2016. — Comprehensive MBS reference covering prepayment models, structuring, and risk analysis.
3. Brueggeman, W. B. & Fisher, J. D. *Real Estate Finance and Investments*, 16th ed. McGraw-Hill, 2019. — Personal finance perspective on mortgages, including tax implications and investment analysis.
4. Sundaresan, S. *Fixed Income Markets and Their Derivatives*, 4th ed. Academic Press, 2014. — Chapter on MBS and prepayment models with a derivatives perspective.
5. Tuckman, B. & Serrat, A. *Fixed Income Securities*, 3rd ed. Wiley, 2011. — Chapters 20-21 on mortgages and MBS, excellent for connecting mortgage math to broader fixed-income theory.
6. Hull, J. C. *Options, Futures, and Other Derivatives*, 11th ed. Pearson, 2021. — Chapter on interest rate derivatives and mortgage-related products.

### Industry Standards

7. Public Securities Association. "Uniform Practices for the Clearance and Settlement of Mortgage-Backed Securities." PSA Standard Prepayment Model. — The industry standard prepayment benchmark.

### Exam Preparation

8. CFA Institute. *CFA Program Curriculum Level II*, "Fixed Income" volume — MBS and prepayment analysis. Covers PSA model, WAL, and negative convexity.
9. CFA Institute. *CFA Program Curriculum Level I*, "Fixed Income" volume — Fundamentals of amortizing securities and present value of annuities.

### Further Reading

For readers interested in the 2008 financial crisis and the role of mortgage mathematics in it, the following provide excellent context:

10. Lewis, M. *The Big Short*. W.W. Norton, 2010. — Narrative account of how mortgage math (and its misapplication) contributed to the crisis.
11. Gorton, G. *Slapped by the Invisible Hand: The Panic of 2007*. Oxford University Press, 2010. — Academic perspective on the MBS market failure.

### Online Resources

12. Federal Reserve Bank of St. Louis (FRED). Mortgage rate data: [https://fred.stlouisfed.org/series/MORTGAGE30US](https://fred.stlouisfed.org/series/MORTGAGE30US)
13. Consumer Financial Protection Bureau (CFPB). Mortgage guides and rate shopping tools: [https://www.consumerfinance.gov/owning-a-home/](https://www.consumerfinance.gov/owning-a-home/)
14. Securities Industry and Financial Markets Association (SIFMA). MBS market statistics and data.
15. Bloomberg. MBS Analytics and prepayment model documentation. Available to terminal subscribers.